# Knee: relabel the training reports with NVIDIA-hosted LLMs (two gates), CPU only
**What this does, in order**
1. **Gate 1, model bake-off (~10 min).** Every candidate model labels the 58 gold studies (same prompt v1 and few-shot examples as your Qwen3-8B run, so the numbers are comparable). The two best usable models are chosen automatically.
2. **Gate 2, full relabel (~4 h at 36 requests/min, shared across both models).** The two chosen models label all 4,407 reports. Resumable: results are cached per model in `nv_*.jsonl`.
3. **Fusion.** Each model's verdicts (present / absent / uncertain / not_mentioned) are mapped to gold-calibrated probabilities, averaged across models, and turned into `labels_soft_v7.csv` (`y_<label>`, `w_<label>`, `is_gold`, `lang`; gold rows keep weight 1).
4. **Go / no-go.** The fused labels are scored against the 58 gold studies with cross-fitted calibration and compared with your old Qwen3-8B labels. It prints GO, WEAK or NO-GO for spending ~8 h of GPU on a retrain.

**Setup.** Kaggle: Add-ons > Secrets > add `NVIDIA_API_KEY` and attach it to this notebook. Internet ON, **no GPU needed** (use a CPU session). Inputs: the competition data, the `knee-mri-pipeline` dataset (v6_3), and (optional but recommended) the old label notebook output (`B.csv`) and the caching output (`labels_soft.csv`) for comparison and fallback. Save & Run All.
**Heads-up:** this sends competition report text to NVIDIA's hosted endpoints. Check the competition Rules on external services before running. The free tier is limited to about 40 requests/min per key, shared across all models, so the run time is set by that limit, not by the models.

In [1]:
import os, sys, re, json, time, threading, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd

W = Path(os.environ.get("KNEE_WORK", "/kaggle/working")); W.mkdir(parents=True, exist_ok=True)
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
def _find_code():
    if os.environ.get("KNEE_CODE"): return Path(os.environ["KNEE_CODE"])
    hits = []
    for d, dirs, fs in os.walk(ROOT):
        dirs[:] = [x for x in dirs if x not in {"train_series", "test_series", "cache", "__pycache__", ".git"}]
        if "labeler_llm.py" in fs and "labeler_combine.py" in fs and "common.py" in fs: hits.append(Path(d))
    print("pipeline code folders found:", *hits, sep="\n   ")
    assert hits, ("pipeline code not found under /kaggle/input: attach the `knee-mri-pipeline` dataset (Add Input -> your datasets) "
                  "and make sure its newest version contains labeler_llm.py, labeler_combine.py and common.py")
    assert len(hits) == 1, f"several code folders, set KNEE_CODE to one of them: {hits}"
    return hits[0]
CODE = _find_code()
assert (CODE / "labeler_llm.py").exists() and (CODE / "labeler_combine.py").exists(), f"pipeline code not found at {CODE}"
sys.path.insert(0, str(CODE))
from common import LABELS, find_data_dir
import labeler_llm as LL, labeler_combine as LC
from sklearn.metrics import roc_auc_score

# ---------------- settings ----------------
BASE_URL = os.environ.get("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
RPM        = float(os.environ.get("RELABEL_RPM", 36))       # free tier is ~40 RPM per key, shared by all models; keep a margin
WORKERS    = int(os.environ.get("RELABEL_WORKERS", 16))
MAX_TOKENS = 1500                                           # JSON answer needs ~250; headroom in case a model ignores "no thinking"
MAX_HOURS_GATE2 = float(os.environ.get("RELABEL_MAX_HOURS", 5.0))   # stop labelling after this long, fuse what exists
NUM_MODELS = 2
CANDIDATES_OVERRIDE = None        # e.g. ["nvidia/nemotron-3-ultra-550b-a55b", "deepseek-ai/deepseek-v4.1-flash"]
FORCE_MODELS = None               # skip the bake-off selection and use exactly these model ids
MAX_CANDIDATES = 6
NM_W, UNC_W = 0.6, 0.5            # weight factors: both models say not_mentioned / any model says uncertain
# keywords used to find model ids in the account's model list (ids on build.nvidia.com change; the list is read live)
KEYS = [("nemotron-3-ultra",), ("deepseek-v4",), ("qwen3.5",), ("kimi-k2",), ("glm-5",), ("gpt-oss-120b",), ("step-5",), ("llama-3.1-nemotron-ultra",)]
EXCLUDE = ("vision", "embed", "rerank", "guard", "safety", "reward", "coder", "code", "parse", "clip", "-vl-", "diffusion")

def get_key():
    k = os.environ.get("NVIDIA_API_KEY")
    if k: return k
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("NVIDIA_API_KEY")
    except Exception:
        return None
KEY = get_key(); assert KEY, "NVIDIA_API_KEY not found: add it as a Kaggle secret (Add-ons > Secrets) and attach it to this notebook"
print("API key: present (not printed) | base url:", BASE_URL, "| RPM", RPM, "| workers", WORKERS)

try:
    import openai
except ImportError:
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True); import openai
from openai import OpenAI

DATA = find_data_dir(); train = pd.read_csv(DATA / "train.csv")
gold_m = LC.gold_mask(train).values; gold = train[gold_m].reset_index(drop=True)
print(f"train studies {len(train)} | gold {len(gold)} | data {DATA}")

# optional inputs: old B.csv (Qwen3-8B verdicts) and old labels_soft.csv
def find(name):
    hits = []
    for d, dirs, fs in os.walk(ROOT):
        dirs[:] = [x for x in dirs if x not in {"cache", "train_series", "test_series", "train", "test", "__pycache__"}]
        if name in fs: hits.append(Path(d) / name)
    return hits
OLD_B = [p for p in find("B.csv")]; OLD_LAB = find("labels_soft.csv")
print("old B.csv:", OLD_B, "| old labels_soft.csv:", OLD_LAB)
OLD_B = OLD_B[0] if len(OLD_B) == 1 else None
OLD_LAB = [p for p in OLD_LAB if "y_ACL" in pd.read_csv(p, nrows=1).columns]
OLD_LAB = OLD_LAB[0] if len(OLD_LAB) == 1 else None
# resume: restore caches written by a previous run of this notebook
for d, dirs, fs in os.walk(ROOT):
    dirs[:] = [x for x in dirs if x not in {"cache", "train_series", "test_series", "train", "test"}]
    for f in fs:
        if f.startswith("nv_") and f.endswith(".jsonl") and not (W / f).exists():
            shutil.copy(Path(d) / f, W / f); print("restored cache", f)

pipeline code folders found:
   /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline
API key: present (not printed) | base url: https://integrate.api.nvidia.com/v1 | RPM 36.0 | workers 16
train studies 4407 | gold 58 | data /kaggle/input/competitions/rsna-knee-abnormality-detection
old B.csv: [] | old labels_soft.csv: []


In [2]:
# ---------------- API plumbing: global rate limiter, retries, per-model caches ----------------
class Limiter:
    def __init__(self, rpm): self.gap, self.lock, self.next = 60.0 / rpm, threading.Lock(), time.time()
    def wait(self):
        with self.lock:
            now = time.time(); t = max(self.next, now); self.next = t + self.gap
        time.sleep(max(0.0, t - now))
    def slow(self, secs):
        with self.lock: self.next = max(self.next, time.time() + secs)
LIM = Limiter(RPM)
client = OpenAI(base_url=BASE_URL, api_key=KEY, timeout=180, max_retries=0)
NOTHINK = {}                                  # model -> still sending the "no thinking" flag?
slug = lambda m: re.sub(r"[^A-Za-z0-9]+", "_", m).strip("_")
cache_path = lambda m: W / f"nv_{slug(m)}.jsonl"

def last_json(text):
    """parse_json takes the first '{' to the last '}'; reasoning text may contain braces, so also try the last block holding a label key."""
    out, ok = LL.parse_json(text)
    if ok: return out, ok
    for blk in reversed(re.findall(r"\{[^{}]*\"ACL\"[^{}]*\}", text or "", flags=re.S)):
        out, ok = LL.parse_json(blk)
        if ok: return out, ok
    return out, False

def call_model(model, report):
    msgs = [{"role": "system", "content": LL.SYSTEM}] + LL.build_messages(report)
    t0 = time.time()
    for attempt in range(7):
        LIM.wait()
        kw = dict(model=model, messages=msgs, temperature=0.0, max_tokens=MAX_TOKENS)
        if NOTHINK.get(model, True): kw["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False, "thinking": False}}
        try:
            r = client.chat.completions.create(**kw)
            m = r.choices[0].message; txt = m.content or ""
            if not txt.strip():
                txt = getattr(m, "reasoning_content", None) or (getattr(m, "model_extra", None) or {}).get("reasoning_content", "") or ""
            return txt, r.choices[0].finish_reason, time.time() - t0
        except openai.BadRequestError as e:
            if "extra_body" in kw and NOTHINK.get(model, True): NOTHINK[model] = False; continue
            return "", "bad_request:" + str(e)[:120], time.time() - t0
        except (openai.NotFoundError, openai.PermissionDeniedError, openai.AuthenticationError) as e:
            return "", "unavailable:" + type(e).__name__, time.time() - t0
        except openai.RateLimitError:
            LIM.slow(15 + 10 * attempt)
        except Exception:
            time.sleep(min(60, 2 ** attempt))
    return "", "failed", time.time() - t0

def load_cache(model):
    done, p = {}, cache_path(model)
    if p.exists():
        for line in p.read_text().splitlines():
            try:
                r = json.loads(line)
                if r.get("parse_ok"): done[r["StudyInstanceUID"]] = r
            except Exception: pass
    return done

LOCKS = {}
def run_tasks(pairs, deadline, label="run"):
    """pairs = [(StudyInstanceUID, report, model)]; skips parsed rows already cached; stops at the deadline. Returns stats."""
    done = {m: load_cache(m) for m in {m for _, _, m in pairs}}
    todo = [(s, r, m) for s, r, m in pairs if s not in done[m]]
    print(f"[{label}] {len(pairs)} tasks | cached {len(pairs) - len(todo)} | to call {len(todo)} | ETA {len(todo) / RPM / 60:.2f} h at {RPM:.0f} RPM")
    files = {m: open(cache_path(m), "a") for m in done}
    flock, stats, t0 = threading.Lock(), dict(ok=0, fail=0, n=0), time.time()
    def work(s, rep, m):
        if time.time() > deadline: return None
        txt, fin, lat = call_model(m, str(rep)[:6000])
        parsed, ok = last_json(txt)
        rec = {"StudyInstanceUID": s, "parse_ok": ok, "finish": fin, "lat": round(lat, 2), **parsed}
        with flock:
            files[m].write(json.dumps(rec, ensure_ascii=False) + "\n"); files[m].flush()
            stats["ok" if ok else "fail"] += 1; stats["n"] += 1
            if stats["n"] % 200 == 0:
                el = time.time() - t0; print(f"  [{label}] {stats['n']}/{len(todo)} | fail {stats['fail']} | {stats['n'] / el * 60:.1f}/min | {el / 3600:.2f} h", flush=True)
        return rec
    with ThreadPoolExecutor(WORKERS) as ex:
        for f in as_completed([ex.submit(work, *t) for t in todo]): f.result()
    for f in files.values(): f.close()
    stats["skipped_deadline"] = len(todo) - stats["n"]
    print(f"[{label}] done: {stats}")
    return stats

def verdict_frame(model, cache):
    """B.csv-style frame (B_, Bc_, Bv_, B_parse_ok) from a model's cache, so labeler_combine helpers can be reused."""
    rows = []
    for s, r in cache.items():
        row = {"StudyInstanceUID": s, "B_parse_ok": True}
        for l in LABELS:
            v = r[l]; row[f"B_{l}"] = LL.PROB[v]; row[f"Bc_{l}"] = LL.CONF[v]; row[f"Bv_{l}"] = v
        rows.append(row)
    return pd.DataFrame(rows)

def gold_auc(frame, col_fmt, min_each=3):
    """per-label AUC vs gold for any frame with StudyInstanceUID + col_fmt.format(l) score columns"""
    g = gold.merge(frame, on="StudyInstanceUID")
    res = {}
    for l in LABELS:
        y = g[l].astype(int).values
        res[l] = roc_auc_score(y, g[col_fmt.format(l)]) if y.sum() >= min_each and (1 - y).sum() >= min_each else np.nan
    return pd.Series(res), len(g)
print("plumbing ready")

plumbing ready


In [3]:
# ---------------- GATE 1: model discovery + gold bake-off ----------------
avail = sorted(m.id for m in client.models.list().data)
print(f"{len(avail)} models visible to this key")
if CANDIDATES_OVERRIDE: cands = list(CANDIDATES_OVERRIDE)
else:
    cands = []
    for toks in KEYS:
        hits = [m for m in avail if all(t in m.lower() for t in toks) and not any(x in m.lower() for x in EXCLUDE)]
        hits.sort(key=lambda m: (any(x in m.lower() for x in ("flash", "lite", "mini", "nano")), len(m)))
        if hits: cands.append(hits[0])
    cands = cands[:MAX_CANDIDATES]
print("candidates:", cands)
assert cands, "no candidate models found: set CANDIDATES_OVERRIDE using ids from the printed model list"
if len(cands) < 2: print("WARNING: fewer than 2 candidates; add ids via CANDIDATES_OVERRIDE")

t_g1 = time.time()
pairs = [(s, r, m) for m in cands for s, r in zip(gold.StudyInstanceUID, gold.Report)]
run_tasks(pairs, deadline=time.time() + 45 * 60, label="gate1")
rows, per_label = [], {}
for m in cands:
    c = load_cache(m); allrecs = [json.loads(x) for x in cache_path(m).read_text().splitlines()] if cache_path(m).exists() else []
    gold_recs = [r for r in allrecs if r["StudyInstanceUID"] in set(gold.StudyInstanceUID)]
    parse = np.mean([r["parse_ok"] for r in gold_recs]) if gold_recs else 0.0
    fr = verdict_frame(m, {s: r for s, r in c.items() if s in set(gold.StudyInstanceUID)})
    if len(fr):
        a, n = gold_auc(fr, "B_{}"); per_label[m] = a
        rows.append(dict(model=m, n=n, parse_rate=round(parse, 3), gold_auc_macro=round(a.mean(), 4), median_latency_s=round(float(np.median([r["lat"] for r in gold_recs])), 1),
                         nothink_flag=NOTHINK.get(m, True)))
    else:
        rows.append(dict(model=m, n=0, parse_rate=0.0, gold_auc_macro=np.nan, median_latency_s=np.nan, nothink_flag=NOTHINK.get(m, True)))
G1 = pd.DataFrame(rows).sort_values("gold_auc_macro", ascending=False).reset_index(drop=True)
ref_auc = np.nan
if OLD_B is not None:
    ob = pd.read_csv(OLD_B); ob = ob[ob.B_parse_ok.astype(bool)] if "B_parse_ok" in ob else ob
    a, n = gold_auc(ob, "B_{}"); per_label["OLD Qwen3-8B (B.csv)"] = a; ref_auc = a.mean()
print(f"Gate 1 took {(time.time() - t_g1) / 60:.1f} min. Gold macro AUC (fixed verdict map, same as your earlier B numbers; n={len(gold)}, SE about 0.02):\n", G1.to_string(index=False))
print("reference, your old Qwen3-8B B on the same gold:", None if np.isnan(ref_auc) else round(ref_auc, 4), "(your handoff notes quote 0.842)")
print("\nper-label gold AUC:\n", pd.DataFrame(per_label).round(3).to_string())

# selection: usable = parses >= 97% and median latency <= 45 s; best first, then best usable model from a DIFFERENT publisher unless it is >0.015 worse than the runner-up
use = G1[(G1.parse_rate >= 0.97) & (G1.median_latency_s <= 45) & G1.gold_auc_macro.notna()].reset_index(drop=True)
if FORCE_MODELS: chosen = list(FORCE_MODELS)
else:
    chosen = [use.model[0]] if len(use) else []
    rest = use.iloc[1:]
    if len(rest):
        diff = rest[rest.model.map(lambda m: m.split("/")[0]) != chosen[0].split("/")[0]]
        pick = diff.iloc[0] if len(diff) and diff.gold_auc_macro.iloc[0] >= rest.gold_auc_macro.iloc[0] - 0.015 else rest.iloc[0]
        chosen.append(pick.model)
    chosen = chosen[:NUM_MODELS]
assert chosen, "no usable model passed the gate (parse rate >= 97%, latency <= 45 s): check the table above / the API key / model ids"
print("\nCHOSEN FOR THE FULL RUN:", chosen)
if len(chosen) < NUM_MODELS: print("WARNING: only", len(chosen), "usable model(s); the fused labels will use a single labeller")
n_calls = len(train) * len(chosen) - sum(len(load_cache(m)) for m in chosen)
print(f"full run: about {n_calls} calls -> {n_calls / RPM / 60:.1f} h at {RPM:.0f} RPM")

81 models visible to this key
candidates: ['nvidia/nemotron-3-ultra-550b-a55b', 'deepseek-ai/deepseek-v4.1-flash', 'moonshotai/kimi-k2.6', 'z-ai/glm-5.3', 'stepfun-ai/step-5-preview', 'nvidia/llama-3.1-nemotron-ultra-253b-v1']
[gate1] 348 tasks | cached 0 | to call 348 | ETA 0.16 h at 36 RPM
  [gate1] 200/348 | fail 65 | 17.3/min | 0.19 h
[gate1] done: {'ok': 189, 'fail': 159, 'n': 348, 'skipped_deadline': 0}
Gate 1 took 21.8 min. Gold macro AUC (fixed verdict map, same as your earlier B numbers; n=58, SE about 0.02):
                                   model  n  parse_rate  gold_auc_macro  median_latency_s  nothink_flag
                           z-ai/glm-5.3 37       0.638          0.8320             126.8          True
              stepfun-ai/step-5-preview 36       0.621          0.8316              74.3          True
      nvidia/nemotron-3-ultra-550b-a55b 58       1.000          0.8304              24.1          True
        deepseek-ai/deepseek-v4.1-flash 58       1.000         

In [4]:
# ---------------- GATE 2: full relabel with the chosen models ----------------
order = [(s, r, m) for s, r in zip(train.StudyInstanceUID, train.Report) for m in chosen]      # study by study, so partial runs stay balanced
t_g2 = time.time()
stats = run_tasks(order, deadline=time.time() + MAX_HOURS_GATE2 * 3600, label="gate2")
caches = {m: load_cache(m) for m in chosen}
for m in chosen: print(f"{m}: {len(caches[m])}/{len(train)} studies labelled")
covered = set.intersection(*[set(c) for c in caches.values()])
print("studies covered by ALL chosen models:", len(covered), "of", len(train), f"| wall {(time.time() - t_g2) / 3600:.2f} h")
if stats.get("skipped_deadline", 0): print("WARNING: stopped at the time limit; re-run this cell (or the notebook) to continue from the caches")

[gate2] 4407 tasks | cached 58 | to call 4349 | ETA 2.01 h at 36 RPM
  [gate2] 200/4349 | fail 0 | 34.5/min | 0.10 h
  [gate2] 400/4349 | fail 0 | 33.6/min | 0.20 h
  [gate2] 600/4349 | fail 0 | 33.9/min | 0.29 h
  [gate2] 800/4349 | fail 0 | 34.5/min | 0.39 h
  [gate2] 1000/4349 | fail 0 | 34.5/min | 0.48 h
  [gate2] 1200/4349 | fail 0 | 34.6/min | 0.58 h
  [gate2] 1400/4349 | fail 0 | 34.6/min | 0.67 h
  [gate2] 1600/4349 | fail 0 | 34.5/min | 0.77 h
  [gate2] 1800/4349 | fail 0 | 34.1/min | 0.88 h
  [gate2] 2000/4349 | fail 0 | 34.2/min | 0.97 h
  [gate2] 2200/4349 | fail 0 | 34.2/min | 1.07 h
  [gate2] 2400/4349 | fail 0 | 34.2/min | 1.17 h
  [gate2] 2600/4349 | fail 0 | 34.3/min | 1.27 h
  [gate2] 2800/4349 | fail 0 | 34.1/min | 1.37 h
  [gate2] 3000/4349 | fail 0 | 34.1/min | 1.47 h
  [gate2] 3200/4349 | fail 0 | 34.0/min | 1.57 h
  [gate2] 3400/4349 | fail 0 | 34.1/min | 1.66 h
  [gate2] 3600/4349 | fail 0 | 34.2/min | 1.76 h
  [gate2] 3800/4349 | fail 0 | 34.2/min | 1.85 h
  [g

In [5]:
# ---------------- fusion, gold evaluation, go/no-go, output ----------------
VER = LC.VERDICTS
frames = {m: verdict_frame(m, caches[m]) for m in chosen}
gold_ids = gold.StudyInstanceUID.values
rng = np.random.default_rng(0); gfold = dict(zip(gold_ids, rng.permutation(len(gold_ids)) % 5))

def apply_table(fr, table):
    out = fr[["StudyInstanceUID"]].copy()
    for l in LABELS: out[f"P_{l}"] = fr[f"Bv_{l}"].map(table[l]).values
    return out

def crossfit_gold(fr_by_model):
    """honest gold score: tables fitted on 4/5 of the gold studies, applied to the held-out 1/5; then averaged over models"""
    parts = []
    for k in range(5):
        te = [s for s in gold_ids if gfold[s] == k]; tr = train[train.StudyInstanceUID.isin([s for s in gold_ids if gfold[s] != k])]
        per = []
        for m, fr in fr_by_model.items():
            tab, _ = LC.fit_verdict_probs(tr, fr[fr.StudyInstanceUID.isin(tr.StudyInstanceUID)])
            per.append(apply_table(fr[fr.StudyInstanceUID.isin(te)], tab).set_index("StudyInstanceUID"))
        parts.append(sum(per) / len(per))
    return pd.concat(parts).reset_index()

cf_fused = crossfit_gold(frames); a_fused, n_g = gold_auc(cf_fused, "P_{}")
rows = {"fused (" + " + ".join(m.split("/")[-1] for m in chosen) + ")": a_fused}
for m in chosen: rows[m.split("/")[-1]] = gold_auc(crossfit_gold({m: frames[m]}), "P_{}")[0]
ref_cf = None
if OLD_B is not None:
    ob = pd.read_csv(OLD_B); ob = ob[ob.B_parse_ok.astype(bool)] if "B_parse_ok" in ob else ob
    rows["OLD Qwen3-8B"] = gold_auc(crossfit_gold({"old": ob}), "P_{}")[0]; ref_cf = rows["OLD Qwen3-8B"].mean()
T = pd.DataFrame(rows); T.loc["MACRO"] = T.mean()
print(f"Cross-fitted calibrated gold AUC (n={n_g} gold studies, 5 folds; SE about 0.02, so treat as a sanity check):\n", T.round(3).to_string())

fused_score = a_fused.mean()
ref_score = ref_cf if ref_cf is not None else 0.842
gain = fused_score - ref_score
verdict = "GO" if gain >= 0.02 else ("WEAK" if gain >= 0.005 else "NO-GO")
print(f"\nfused {fused_score:.4f} vs old labels {ref_score:.4f}{'' if ref_cf is not None else ' (constant from your handoff; attach old B.csv for a like-for-like number)'}  ->  gain {gain:+.4f}  ->  {verdict}")
print({"GO": "Retrain on labels_soft_v7.csv: the new labels agree with gold clearly better.",
       "WEAK": "Small, within-noise improvement. Retrain only if you have nothing better for the GPU time (seed 43 is the safe alternative).",
       "NO-GO": "No real agreement gain. Do NOT spend the retrain on these labels; run seed 43 instead."}[verdict])

# ---- apply to all studies: tables fitted on ALL gold, one per model ----
tabs = {m: LC.fit_verdict_probs(train, frames[m])[0] for m in chosen}
P = {m: apply_table(frames[m], tabs[m]).set_index("StudyInstanceUID") for m in chosen}
sids = train.StudyInstanceUID.values
lang = train.Report.fillna("").map(LC.detect_lang).values
out = pd.DataFrame({"StudyInstanceUID": sids, "lang": lang, "is_gold": gold_m})
fused_cols = {}
for l in LABELS:
    arr = np.stack([P[m][f"P_{l}"].reindex(sids).values for m in chosen], 1)          # [N, n_models], NaN where not labelled
    verd = np.stack([frames[m].set_index("StudyInstanceUID")[f"Bv_{l}"].reindex(sids).values for m in chosen], 1)
    y = np.nanmean(arr, 1)
    spread = (np.nanmax(arr, 1) - np.nanmin(arr, 1)) if arr.shape[1] > 1 else np.zeros(len(sids))
    agree = 1 - np.nan_to_num(spread, nan=0.2)
    all_nm = np.all((verd == "not_mentioned") | pd.isna(verd), 1); any_unc = np.any(verd == "uncertain", 1)
    factor = np.where(all_nm, NM_W, np.where(any_unc, UNC_W, 1.0))
    a = a_fused[l]; base = float(np.clip(0.7 * max(0.0, (a - 0.5) * 2), LC.W_MIN, LC.W_MAX)) if a == a else LC.DEFAULT_BASE_W
    w = base * (0.5 + 0.5 * agree) * factor
    gv = train[l].values
    y = np.where(gold_m, gv, y); w = np.where(gold_m, 1.0, w)
    part = (~gold_m) & ~pd.isna(gv); y = np.where(part, gv, y); w = np.where(part, 1.0, w)
    fused_cols[l] = (y, w)
nan_rows = np.zeros(len(sids), bool)
for l in LABELS: nan_rows |= np.isnan(fused_cols[l][0])
print("studies without a fused label (not labelled by every chosen model):", int(nan_rows.sum()))
if nan_rows.any():
    if OLD_LAB is not None:
        old = pd.read_csv(OLD_LAB).set_index("StudyInstanceUID").reindex(sids)
        for l in LABELS:
            y, w = fused_cols[l]; m_ = np.isnan(y)
            y = np.where(m_, old[f"y_{l}"].values, y); w = np.where(m_, old[f"w_{l}"].values, w); fused_cols[l] = (y, w)
        print("  -> filled from the OLD labels_soft.csv")
    else:
        for l in LABELS:
            y, w = fused_cols[l]; fused_cols[l] = (np.where(np.isnan(y), 0.05, y), np.where(np.isnan(y), 0.0, w))
        print("  -> filled with y=0.05, weight 0 (they do not train); attach the old labels_soft.csv to fall back to it instead")
for l in LABELS: out[f"y_{l}"] = fused_cols[l][0].astype(float)
for l in LABELS: out[f"w_{l}"] = fused_cols[l][1].astype(float)
out.to_csv(W / "labels_soft_v7.csv", index=False)
T.round(4).to_csv(W / "relabel_gold_table.csv"); G1.to_csv(W / "gate1_table.csv", index=False)
json.dump(dict(chosen=chosen, fused_gold_auc=float(fused_score), reference=float(ref_score), gain=float(gain), verdict=verdict, n_gold=int(n_g),
               nm_w=NM_W, unc_w=UNC_W, rpm=RPM, labelled={m: len(caches[m]) for m in chosen}), open(W / "relabel_report.json", "w"), indent=1)
chk = pd.DataFrame({"mean_y_new": [out[f"y_{l}"][~gold_m].mean() for l in LABELS], "mean_w_new": [out[f"w_{l}"][~gold_m].mean() for l in LABELS]}, index=LABELS)
if OLD_LAB is not None:
    old = pd.read_csv(OLD_LAB).set_index("StudyInstanceUID").reindex(sids)
    chk["mean_y_old"] = [old[f"y_{l}"][~gold_m].mean() for l in LABELS]; chk["mean_w_old"] = [old[f"w_{l}"][~gold_m].mean() for l in LABELS]
print("\nnon-gold label summary (new vs old):\n", chk.round(3).to_string())
print("\nsaved:", sorted(p.name for p in W.glob("*") if p.suffix in (".csv", ".json", ".jsonl")))

Cross-fitted calibrated gold AUC (n=58 gold studies, 5 folds; SE about 0.02, so treat as a sanity check):
                   fused (nemotron-3-ultra-550b-a55b)  nemotron-3-ultra-550b-a55b
ACL                                            0.925                       0.925
MCL                                            0.862                       0.862
Medial Meniscus                                0.864                       0.864
Lateral Meniscus                               0.824                       0.824
Medial OA                                      0.837                       0.837
Lateral OA                                     0.726                       0.726
PF OA                                          0.748                       0.748
Effusion                                       0.572                       0.572
Synovitis                                      0.587                       0.587
Baker's                                        0.829                       0.829
Co